# Features for B1: what to keep, what to drop

The builder now writes two feature sets for every run: the 13 **base** features (`x`, the ones B0 draws from) and 6
**extra** channels (`x_extra`: sockets, file descriptors, page faults, disk writes, container starts). This notebook asks,
feature by feature, whether a model should keep it.

**Data.** Clean sessions of the train split (fit) and the val split (healthy check), and the dev fault runs (the tuning
fold). The test fold is never read: the choice made here is then evaluated once on it.

**Model units.** Base features as `inputs.prepare` gives them (log1p in their units, memory as its change per step); extra
features as log1p of their raw value. A z-score uses the train mean and deviation, with the deviation floored at `FLOOR`.
`restarts` and `starts` are 0 in healthy runs: they are rules, judged apart.

**The rules, fixed before the numbers.** A feature is dropped when one of these holds:
1. **dead**: flat or absent in every service of the app;
2. **redundant**: the other features of its service predict it on val with median R² ≥ `R2_REDUNDANT`, and what they do
   not predict (its residual) never responds to a dev fault. Applied one feature at a time, the most redundant first,
   so that of two twins only one goes;
3. **noise only**: in healthy 15-minute windows it reaches |z| ≥ `Z_ALARM` more often than `NOISY`, and it never
   responds to a dev fault.

A feature that is clean and not redundant but never responds to a dev fault is a **doubt**, not a drop: dev does not hold
every kind of fault. **Responds** means: on the fault's target, after the injection, the feature or its residual
(section 3: what the other features of its service do not predict) reaches |z| ≥ `Z_ALARM` for two steps in a row (the
persistence of the protocol), on average over the repetitions of a dev scenario. `cpu` is exactly `cpu_user` +
`cpu_system`: it is dropped as derived, whatever its numbers.

A feature can also be flat in some services only: that is a per-service mask (the PCA already leaves such cells out), not a drop.

The clean sessions collected before 2026-11-24 have no extra channels (their raw files were lost); they are being
collected again. Until the training sessions have them, the extra channels are **not judged**: their rows stay empty
in the figures and tables, which have no healthy reference for them yet.

In [ ]:
import json
import sys
import warnings
from pathlib import Path
from types import SimpleNamespace

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "anomaly" / "build.py").exists())
sys.path.insert(0, str(REPO))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LogNorm

from anomaly import build, inputs
from anomaly.models import evaluate as ev

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 60)

APP = "hotel"  # "hotel" or "social"
DATA = build.DATASET / "built"
FLOOR = 0.1  # floor of a z-score's deviation, in model units (log1p)
Z_ALARM = 4.0  # |z| that counts as a response
R2_REDUNDANT = 0.95
NOISY = 0.05  # share of healthy 15-minute windows
WIN = 30  # steps of 30 s: 15 minutes, as long as a dev fault's observation

BASE = [f[0] for f in build.FEATURES]
EXTRA = [f[0] for f in build.EXTRA]
EVENTS = ("restarts", "starts")
CONT = [f for f in BASE + EXTRA if f not in EVENTS]
DERIVED = ("cpu",)  # exactly cpu_user + cpu_system: never a predictor, or its parts would look redundant

## 0. The runs

`build.arrays(z, extra=True)` gives the base features followed by the extra ones; `build.arrays(z)` the base ones only.

In [ ]:
def model_units(x, features, exists):
    """[T, S, len(CONT)]: base features through inputs.prepare, extra ones as log1p of their raw value (filled alike)."""
    base, extra = [f for f in CONT if f in BASE], [f for f in CONT if f in EXTRA]
    b = inputs.prepare(x, features, keep=base, exists=exists)
    filled = inputs.fill_misses(x, features, exists)[0]
    e = np.log1p(np.clip(filled[:, :, [features.index(f) for f in extra]], 0, None))
    return np.concatenate([b, e], axis=2)


runs = []
for p in sorted((DATA / APP).glob("*/*.npz")):
    L = json.loads(p.with_suffix(".json").read_text())
    role = L["split"] if p.parent.name == "clean" else p.parent.name
    if role in ("train", "val", "dev"):
        z = np.load(p)
        x, _, F = build.arrays(z, extra=True)
        runs.append(SimpleNamespace(id=f"{p.parent.name}/{p.stem}", role=role, L=L, x=x, t=z["t"], rate=z["rate"]))
        S = [str(s) for s in z["services"]]
exists = np.any([~np.isnan(r.x).all(0) for r in runs if r.role == "train"], axis=0)
for r in runs:
    r.on = (r.rate >= 0) & (r.t > r.L["t_workload"] + ev.WARMUP_S)  # loaded and past the warm-up
    r.m = model_units(r.x, F, exists)
    r.events = np.nan_to_num(r.x[:, :, [F.index(f) for f in EVENTS]])
    if r.role == "dev":
        r.post = r.on & (r.t > r.L["t_inject"])
        r.targets = [S.index(n) for n in (r.L.get("accepted_targets") or [r.L["target"]]) if n in S]
        r.scenario = r.id.split("/")[1].rsplit("__r", 1)[0]
DEV = [r for r in runs if r.role == "dev"]
TR = np.concatenate([r.m[r.on] for r in runs if r.role == "train"])
VA = np.concatenate([r.m[r.on] for r in runs if r.role == "val"])
MU, SD = np.nanmean(TR, axis=0), np.nanstd(TR, axis=0)
LIVE = exists[:, [F.index(f) for f in CONT]] & (SD > 1e-6)  # [S, CONT]: cells that exist and move in training
zscore = lambda m: (m - MU) / np.maximum(SD, FLOOR)


def persistent(a):
    """|a| held for two steps: the min of |a| at t and t-1 (NaN at the first step)."""
    a = np.abs(a)
    return np.minimum(a, np.concatenate([np.full_like(a[:1], np.nan), a[:-1]]))


print(f"{APP}: {len(S)} services; train {sum(r.role == 'train' for r in runs)} sessions ({len(TR)} steps), "
      f"val {sum(r.role == 'val' for r in runs)} ({len(VA)} steps), dev {len(DEV)} runs in {len({r.scenario for r in DEV})} scenarios")

## 1. Inventory

Where each feature exists, where it never moves in training, and how often it is exactly 0.

In [ ]:
spec = {f[0]: f for f in build.FEATURES + build.EXTRA}
raw = np.concatenate([r.x[r.on] for r in runs if r.role == "train"])
rows = []
for f in BASE + EXTRA:
    j, e = F.index(f), exists[:, F.index(f)]
    sd = np.nanstd(raw[:, :, j], axis=0)
    rows.append({"feature": f, "set": "base" if f in BASE else "extra", "in B0": f in inputs.B0, "metric": spec[f][1],
                 "kind": spec[f][2], "services": f"{e.sum()}/{len(S)}", "flat in train": int((e & (sd < 1e-9)).sum()),
                 "zero share": round(float(np.nanmean(raw[:, e, j] == 0)), 3) if e.any() else np.nan})
INVENTORY = pd.DataFrame(rows).set_index("feature")
INVENTORY

## 2. Healthy behaviour

**Figure 1.** Train deviation of every feature in every service, in model units. Grey: the cell does not exist or never
moves. A deviation under `FLOOR` is floored in the z-score, so very small values do not blow it up.

The table: the median over services of the deviation and of |corr| with the load (log1p of the wrk2 rate), and the share
of healthy val windows of 15 minutes where the feature reaches |z| ≥ `Z_ALARM` for two steps (the false responses).

In [ ]:
fig, ax = plt.subplots(figsize=(1 + 0.42 * len(S), 0.36 * len(CONT) + 1.5))
cells = np.where(LIVE, SD, np.nan).T
cmap = plt.get_cmap("Blues").copy()
cmap.set_bad("#d9d9d9")
im = ax.imshow(np.ma.masked_invalid(cells), aspect="auto", cmap=cmap, norm=LogNorm(vmin=1e-3, vmax=3))
ax.set_yticks(range(len(CONT)), [f + ("  (extra)" if f in EXTRA else "") for f in CONT])
ax.set_xticks(range(len(S)), S, rotation=60, ha="right", fontsize=8)
fig.colorbar(im, ax=ax, label="train deviation (model units, log scale)")
ax.set_title(f"{APP}: how much each feature moves in healthy training")
plt.show()

rate_tr = np.concatenate([np.log1p(r.rate[r.on]) for r in runs if r.role == "train"])
hits, seen = np.zeros(len(CONT)), np.zeros(len(CONT))
for r in runs:
    if r.role != "val":
        continue
    pz, idx = persistent(zscore(r.m)), np.flatnonzero(r.on)
    for k in range(0, len(idx) - WIN + 1, WIN):
        top = np.nanmax(pz[idx[k:k + WIN]], axis=0) >= Z_ALARM  # [S, CONT]
        hits += (top & LIVE).sum(0)
        seen += LIVE.sum(0)
healthy = pd.DataFrame({
    "live services": LIVE.sum(0),
    "median train sd": [np.nanmedian(np.where(LIVE[:, j], SD[:, j], np.nan)) for j in range(len(CONT))],
    "median |corr| with load": [np.nanmedian([abs(np.corrcoef(TR[:, i, j], rate_tr)[0, 1]) for i in np.flatnonzero(LIVE[:, j])
                                              if np.isfinite(TR[:, i, j]).all()] or [np.nan]) for j in range(len(CONT))],
    "false responses (val windows)": np.where(seen > 0, hits / np.maximum(seen, 1), np.nan),
}, index=CONT).round(3)
healthy

## 3. Redundancy

Per service, a ridge regression predicts each feature from the other features of the same service, fit on train and
scored on val (out-of-sample R²). High R² in healthy data is not redundancy by itself: a fault can break a pair that
moves together in healthy runs (for example bytes and packets when the responses become errors). So for each feature the
residual, what the others do not predict, is z-scored with its train deviation and checked on the dev faults in section 4.

**Figure 2.** Val R² of each feature from the others, per service (white: below 0; grey: not a live cell).

In [ ]:
def ridge(X, y, lam=1.0):
    mu, sd = X.mean(0), X.std(0) + 1e-9
    Z = (X - mu) / sd
    w = np.linalg.solve(Z.T @ Z + lam * np.eye(Z.shape[1]), Z.T @ (y - y.mean()))
    return lambda X2: ((X2 - mu) / sd) @ w + y.mean()


def redundancy(kept):
    """R² [S, CONT] on val, and per (service, feature) the predictors, the fit and the residual's train deviation."""
    r2, fits = np.full((len(S), len(CONT)), np.nan), {}
    for i in range(len(S)):
        cols = [CONT.index(f) for f in kept if LIVE[i, CONT.index(f)]]
        tr, va = TR[:, i, cols], VA[:, i, cols]
        tr, va = tr[np.isfinite(tr).all(1)], va[np.isfinite(va).all(1)]
        for a, j in enumerate(cols):
            others = [b for b, k in enumerate(cols) if k != j and CONT[k] not in DERIVED]
            if not others or len(tr) < 50:
                continue
            pred = ridge(tr[:, others], tr[:, a])
            r2[i, j] = 1 - np.mean((va[:, a] - pred(va[:, others])) ** 2) / max(np.var(va[:, a]), 1e-12)
            fits[i, j] = ([cols[b] for b in others], pred, max(np.std(tr[:, a] - pred(tr[:, others])), FLOOR))
    return r2, fits


def dev_response(fits=None):
    """[scenario x feature]: on the target, the max after the injection of |z| held two steps, mean over the repetitions.
    With fits: the residual's z instead (what the other features of the service do not predict)."""
    rows = {}
    for r in DEV:
        out = np.full(len(CONT), np.nan)
        pz = persistent(zscore(r.m)) if fits is None else None
        for j in range(len(CONT)):
            best = []
            for i in r.targets:
                if fits is None:
                    if LIVE[i, j]:
                        best.append(np.nanmax(pz[r.post, i, j], initial=0))
                elif (i, j) in fits:
                    others, pred, scale = fits[i, j]
                    X = r.m[:, i, others]
                    res = np.full(len(X), np.nan)
                    ok = np.isfinite(X).all(1) & np.isfinite(r.m[:, i, j])
                    res[ok] = (r.m[ok, i, j] - pred(X[ok])) / scale
                    best.append(np.nanmax(persistent(res)[r.post], initial=0))
            out[j] = max(best) if best else np.nan
        rows.setdefault(r.scenario, []).append(out)
    return pd.DataFrame({k: np.nanmean(v, axis=0) for k, v in rows.items()}, index=CONT).T


R2, FITS = redundancy(CONT)
fig, ax = plt.subplots(figsize=(1 + 0.42 * len(S), 0.36 * len(CONT) + 1.5))
cmap = plt.get_cmap("Purples").copy()
cmap.set_bad("#d9d9d9")
im = ax.imshow(np.ma.masked_invalid(np.clip(R2, 0, 1).T), aspect="auto", cmap=cmap, vmin=0, vmax=1)
ax.set_yticks(range(len(CONT)), CONT)
ax.set_xticks(range(len(S)), S, rotation=60, ha="right", fontsize=8)
fig.colorbar(im, ax=ax, label="val R² from the other features of the service")
ax.set_title(f"{APP}: how predictable each feature is from the rest (healthy)")
plt.show()

# rule 2, one feature at a time: drop the most redundant one whose residual never responds, refit, repeat
kept, DROPPED = list(CONT), []
while True:
    r2, fits = redundancy(kept)
    med = pd.Series(np.nanmedian(r2, axis=0), index=CONT)[kept].dropna()
    unique = dev_response(fits)
    cand = [f for f in med.sort_values(ascending=False).index if med[f] >= R2_REDUNDANT and not (unique[f] >= Z_ALARM).any()]
    if not cand:
        break
    DROPPED.append({"feature": cand[0], "median val R²": round(med[cand[0]], 3), "max unique dev response": round(unique[cand[0]].max(), 1)})
    kept.remove(cand[0])
pd.DataFrame(DROPPED or [{"feature": "none"}]).set_index("feature")

## 4. Response to the dev faults

**Figure 3.** Left: the response of each feature on the target of each dev scenario (max |z| held two steps after the
injection, mean over repetitions). Right: the same for the residual, what the other features of the service do not
predict. Numbers ≥ `Z_ALARM` are a response. A feature with a response on the right carries something the others do not.

In [ ]:
RESP, UNIQUE = dev_response(), dev_response(FITS)
fig, axes = plt.subplots(1, 2, figsize=(2 + 0.7 * len(CONT) * 2, 0.42 * len(RESP) + 2.5), sharey=True)
for ax, table, title in zip(axes, (RESP, UNIQUE), ("response of the feature", "response of its residual")):
    v = table.values
    im = ax.imshow(np.clip(v, 0, 20), aspect="auto", cmap="Oranges", vmin=0, vmax=20)
    for (a, b), val in np.ndenumerate(v):
        if np.isfinite(val):
            ax.text(b, a, f"{val:.0f}", ha="center", va="center", fontsize=7, color="black" if val < 12 else "white",
                    fontweight="bold" if val >= Z_ALARM else "normal")
    ax.set_xticks(range(len(CONT)), CONT, rotation=60, ha="right", fontsize=8)
    ax.set_title(title)
axes[0].set_yticks(range(len(RESP)), RESP.index, fontsize=8)
fig.colorbar(im, ax=axes, label="|z| held two steps (clipped at 20)", shrink=0.8)
fig.suptitle(f"{APP}: dev faults, on their target")
plt.show()

The two rules: how often they fire in healthy steps (train and val, after the warm-up), and on the target of which dev
scenarios they fire after the injection.

In [ ]:
rules = []
for k, f in enumerate(EVENTS):
    healthy_steps = sum(int((r.events[r.on, :, k] > 0).any(1).sum()) for r in runs if r.role in ("train", "val"))
    fired = sorted({r.scenario for r in DEV if (r.events[r.post][:, r.targets, k] > 0).any()})
    rules.append({"rule": f, "set": "base" if f in BASE else "extra", "healthy steps firing": healthy_steps,
                  "dev scenarios where it fires on the target": ", ".join(fired) or "none"})
RULES = pd.DataFrame(rules).set_index("rule")
RULES

## 5. Verdict

The rules of the introduction, applied as written. `max response` and `max unique` are the largest values of figure 3
over the dev scenarios (the feature, its residual); `scenarios` counts those where either responds.

In [ ]:
dropped = {d["feature"]: d for d in DROPPED}
rows = []
for f in CONT:
    j = CONT.index(f)
    resp = np.nanmax([RESP[f].max(), UNIQUE[f].max()])  # the feature or its residual
    if f in EXTRA and not exists[:, F.index(f)].any():
        verdict = "not judged: no training session has it yet"
    elif f in DERIVED:
        verdict = "drop: derived (cpu_user + cpu_system)"
    elif LIVE[:, j].sum() == 0:
        verdict = "drop: dead"
    elif f in dropped:
        verdict = "drop: redundant"
    elif healthy.loc[f, "false responses (val windows)"] > NOISY and not resp >= Z_ALARM:
        verdict = "drop: noise only"
    elif not resp >= Z_ALARM:
        verdict = "doubt: no dev response"
    else:
        verdict = "keep"
    rows.append({"feature": f, "set": "base" if f in BASE else "extra", "in B0": f in inputs.B0,
                 "live services": int(LIVE[:, j].sum()), "median val R²": round(float(np.nanmedian(R2[:, j])), 3),
                 "false responses": healthy.loc[f, "false responses (val windows)"], "max response": round(RESP[f].max(), 1),
                 "max unique": round(UNIQUE[f].max(), 1), "scenarios": int(((RESP[f] >= Z_ALARM) | (UNIQUE[f] >= Z_ALARM)).sum()),
                 "verdict": verdict})
for f, row in RULES.iterrows():
    fires = row["dev scenarios where it fires on the target"] != "none"
    rows.append({"feature": f, "set": row["set"], "in B0": f in inputs.B0,
                 "verdict": ("keep as a rule" if fires else "doubt: never fires on dev") if row["healthy steps firing"] == 0
                 else "check: fires in healthy runs"})
VERDICT = pd.DataFrame(rows).set_index("feature")
VERDICT

## Readings

Written on 2026-11-24 from the dataset of that day: train 15 Hotel and 6 Social clean sessions, val 2 and 2, dev 23 and
25 runs. The extra channels are **not judged** yet: no training session has them until the clean sessions are collected
again (train about 2026-11-27, val about 2026-11-28). Run the notebook again then.

**Hotel**
- **Drop:** `cpu` (derived), `cpu_throttled` (0 in every service: no CPU limit is ever hit), `rx_packets` (the other
  features predict it with val R² 0.999, and its residual peaks at |z| 4.0 on the dev faults, right at the limit).
- **Doubt:** `mem_cache`. It moves in 6 services only (the MongoDBs, deviation 0.004) and never responds to a dev fault.
- **Keep:** `cpu_user`, `cpu_system` (their pair breaks under CPU stress: residual |z| up to 41 and 21), `mem_working_set`,
  `mem_rss` (up to 13 under memory stress), `threads` (12 under pod kill), `rx_bytes`, `tx_bytes` (11 under the bandwidth
  limit), `tx_packets`.
- **Rules:** `restarts` never fires on Hotel's dev faults (none restarts a container: the pod kill replaces the pod),
  but the test has `container_kill`: keep it. `starts` fires on the pod kill only, never in healthy runs: keep it.

**Social**
- **Drop:** `cpu` (derived), `cpu_throttled` (absent: no CPU limits).
- **Keep:** everything else. `mem_cache` responds through its residual (|z| 11 on home-timeline-redis when its bandwidth
  is cut), with 4.5% false responses in healthy val windows, close to the 5% limit. `rx_packets` and `tx_packets`
  respond through their residual under the user-timeline MongoDB pod failure (10 and 20).
- **Rules:** `restarts` fires on the pod failure, `starts` on the pod failure and the pod kill; neither in healthy runs.

**Across the two apps.** `rx_packets` and `mem_cache` get different verdicts: Hotel would drop them, Social keeps them.
One model per app may use its own set; one feature set for both apps would keep them.

**How the rules were applied.** A first run judged "responds" on the feature alone, which flagged `cpu_system` (both
apps) and Social's `mem_cache`, `rx_packets` and `tx_packets` as doubts although their residuals respond strongly; it
also left `cpu` to the redundancy rule, which a sum in log space escapes. "Responds" was aligned with rule 2 (the
feature or its residual) and `cpu` was marked as derived, as stated in the introduction.